# Instituto Tecnológico y de Estudios Superiores de Monterrey

## Análisis de Grandes Volúmenes de Datos

### Actividad 4 | Métricas de calidad de resultados

**Profesor:** Dr. Iván Olmos Pineda  
**Fecha de entrega:** 14 de junio de 2026  
**Alumno:** Hiram García Austria  
**Matricula:** A0038771

---

## Objetivo de la actividad

Identificar métricas para la medición de la calidad de resultados derivados de la aplicación de modelos de aprendizaje, ya sea supervisado o no supervisado, orientado al procesamiento de grandes volúmenes de datos, que permitan la selección de los modelos que mejor se ajusten a la tarea de aprendizaje a resolver.

---

## Contexto del proyecto

Como parte del proyecto del curso, se seleccionó un dataset histórico del mercado bursátil que contiene información de más de 9,000 acciones con registros diarios desde 1962.

Este conjunto de datos incluye variables financieras clave como precios de apertura, cierre, máximos, mínimos, volumen de transacciones, dividendos y ajustes por división de acciones, representando más de 34 millones de registros y un tamaño aproximado de 4.47 GB.

La población objetivo corresponde al universo completo de registros históricos contenidos en el dataset bursátil seleccionado.

---

## Selección de los datos

A continuación se recolecta una muestra de dimensión contenida de la base de datos (D). Para ello se obtienen particiones que cumplan con los criterios de las variables de caracterización identificadas:
* Periodo económico  
* Volumen de transacciones
* Nivel de volatilidad

Después se obtendrán un número limitado de instancias de cada partición aplicando la técnica de **muestreo estratificado**, lo que permitirá construir una muestra (M) a partir de la unión de las instancias que se recuperan de este proceso.

In [1]:
import os
import findspark
import kagglehub

findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import expr, count, when, col, to_date, lit, round, concat_ws

spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Property used to format output tables better spark
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

c:\Users\hille\anaconda3\envs\env_pyspark\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download latest version
path = kagglehub.dataset_download("jakewright/9000-tickers-of-stock-market-data-full-history")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\hille\.cache\kagglehub\datasets\jakewright\9000-tickers-of-stock-market-data-full-history\versions\2


In [3]:
path = path + "/all_stock_data.csv"
print(path)

C:\Users\hille\.cache\kagglehub\datasets\jakewright\9000-tickers-of-stock-market-data-full-history\versions\2/all_stock_data.csv


In [4]:
# Definimos el esquema del data set
schema = StructType([
    StructField("Date", DateType(), True),
    StructField("Ticker", StringType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Volume", DoubleType(), True),
    StructField("Dividends", DecimalType(5,1), True),
    StructField("Stock Splits", DecimalType(5,1), True)
])

In [5]:
# Disparador 0
# Leemos el data set e imprimimos los primeros 5 registros
df = spark.read.csv(path, header=True, inferSchema=False, schema=schema)
df.show(5)

+----------+------+----+-------------------+-------------------+-------------------+---------+---------+------------+
|      Date|Ticker|Open|               High|                Low|              Close|   Volume|Dividends|Stock Splits|
+----------+------+----+-------------------+-------------------+-------------------+---------+---------+------------+
|1962-01-02|    ED| 0.0| 0.2658275556233194|0.26178762316703796|0.26178762316703796|  25600.0|      0.0|         0.0|
|1962-01-02|   CVX| 0.0|0.04680890217423439|0.04606926600933256|0.04680890217423439| 105840.0|      0.0|         0.0|
|1962-01-02|    GD| 0.0|0.21003275954390174|0.20306070787008793| 0.2082897424697876|2648000.0|      0.0|         0.0|
|1962-01-02|    BP| 0.0|0.14143933090345925|0.13952797651290894|0.13952797651290894|  77440.0|      0.0|         0.0|
|1962-01-02|   MSI| 0.0| 0.7649229763450202| 0.7452535214492476| 0.7518101930618286|  65671.0|      0.0|         0.0|
+----------+------+----+-------------------+------------

In [6]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Ticker: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Dividends: decimal(5,1) (nullable = true)
 |-- Stock Splits: decimal(5,1) (nullable = true)



In [7]:
# Definimos el cache
df.cache()
resumen = df.describe()
valores_nulos = df.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
])

In [8]:
# Disparador 1
num = df.count()

# Imprimimos el numero de columnas y registros
print(f"Número de columnas: {len(df.columns)}")
print(f"Número de registros: {num:,}\n")

Número de columnas: 9
Número de registros: 34,646,258



In [9]:
# Disparador 2
resumen_transpuesto = resumen.toPandas().set_index('summary').T
# Imprimimos el  resumen
resumen_transpuesto

summary,count,mean,stddev,min,max
Ticker,34646258,NaN,NaN,A,ZZLL
Open,34646149,1.1150488437326563E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
High,34646149,1.1150488437326608E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
Low,34646149,1.115048843732678E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
Close,34646152,1.1150487471809343E23,3.95473476978514E25,-8.210044423351318E25,1.5072897744279783E28
Volume,34646258,1339229.8683998429,1.5671698842802074E7,0.0,9.230856E9
Dividends,34646258,0.00401,1.6035252555952273,0.0,4500.0
Stock Splits,34646255,0.00039,0.1896345452067494,0.0,1000.0


In [10]:
# Disparador 3
nulos_pandas = valores_nulos.toPandas().T

# Imprimimos los valores nulos
nulos_pandas.columns = ['Valores nulos']
nulos_pandas["%"] = (nulos_pandas["Valores nulos"] / num) * 100
nulos_pandas

,Valores nulos,%
Date,0,0.000000
Ticker,0,0.000000
Open,109,0.000315
High,109,0.000315
Low,109,0.000315
Close,106,0.000306
Volume,0,0.000000
Dividends,0,0.000000
Stock Splits,3,0.000009


## Estrategia de Particionamiento

Vamos a implementar la estrategia de particionamiento descrita, utilizando las variables **Periodo económico**, **Nivel de volatilidad**, y **Volumen de transacciones**.

### Variable: Periodo Económico
Primero, definiremos una función para clasificar las fechas en 'Pre-crisis', 'Crisis' o 'Recuperación' basándonos en los eventos macroeconómicos mencionados.

In [11]:
# Definición de periodos económicos relevantes para segmentación del dataset
crisis_periods = [
    ("1987-10-01", "1988-03-31", "Crisis"), # Lunes negro
    ("2000-03-01", "2001-11-30", "Crisis"), # Burbuja de las punto com
    ("2008-09-01", "2009-03-31", "Crisis"), # Crisis financiera
    ("2020-02-01", "2020-05-31", "Crisis"), # Pandemia del Covid
    ("2022-02-01", "2023-01-31", "Crisis")  # Guerra Rusia vs Ucrania (fin arbitrario para el ejemplo)
]

# Definir periodos de recuperación (aproximados)
recovery_periods = [
    ("1988-04-01", "2000-02-29", "Recuperación"), # Después de Lunes negro, antes de Dot-com
    ("2001-12-01", "2008-08-31", "Recuperación"), # Después de Dot-com, antes de Crisis financiera
    ("2009-04-01", "2020-01-31", "Recuperación"), # Después de Crisis financiera, antes de Covid
    ("2020-06-01", "2022-01-31", "Recuperación"), # Después de Covid, antes de Guerra
    ("2023-02-01", "2024-12-31", "Recuperación")  # Después de Guerra (arbitrario para el futuro)
]

# Inicializar la columna 'Periodo_Economico' con 'Pre-crisis' como valor por defecto
df_partitioned = df.withColumn("Periodo_Economico", lit("Pre-crisis"))

# Aplicar las condiciones para Periodos de Crisis
for start_date_str, end_date_str, period_type in crisis_periods:
    start_date = to_date(lit(start_date_str))
    end_date = to_date(lit(end_date_str))
    df_partitioned = df_partitioned.withColumn("Periodo_Economico",
                                                when((col("Date") >= start_date) & (col("Date") <= end_date), lit(period_type))
                                                .otherwise(col("Periodo_Economico")))

# Aplicar las condiciones para Periodos de Recuperación
for start_date_str, end_date_str, period_type in recovery_periods:
    start_date = to_date(lit(start_date_str))
    end_date = to_date(lit(end_date_str))
    df_partitioned = df_partitioned.withColumn("Periodo_Economico",
                                                when((col("Date") >= start_date) & (col("Date") <= end_date), lit(period_type))
                                                .otherwise(col("Periodo_Economico")))

# Mostrar la distribución de los periodos económicos
df_partitioned.groupBy("Periodo_Economico").count().show()

+-----------------+--------+
|Periodo_Economico|   count|
+-----------------+--------+
|       Pre-crisis| 1560048|
|           Crisis| 4206011|
|     Recuperación|28880199|
+-----------------+--------+



### Variable: Nivel de Volatilidad

Se calcula la volatilidad diaria como la variación porcentual entre los precios máximo (*High*) y mínimo (*Low*), clasificando posteriormente los registros en niveles de volatilidad: *Baja*, *Media* y *Alta*.

In [12]:
# Mostrar la distribución de los niveles de volatilidad
df_partitioned = df_partitioned.withColumn(
    "Volatilidad_Diaria_Pct",
    expr("try_divide((High - Low) * 100, Low)")
)

df_partitioned = df_partitioned.withColumn(
    "Nivel_Volatilidad",
    when(col("Volatilidad_Diaria_Pct").isNull(), lit("Sin dato"))
    .when(col("Volatilidad_Diaria_Pct") < 2, lit("Baja"))
    .when((col("Volatilidad_Diaria_Pct") >= 2) & (col("Volatilidad_Diaria_Pct") <= 5), lit("Media"))
    .otherwise(lit("Alta"))
)

df_partitioned.groupBy("Nivel_Volatilidad").count().show()

+-----------------+--------+
|Nivel_Volatilidad|   count|
+-----------------+--------+
|             Alta| 7621183|
|            Media|10781440|
|         Sin dato|    1059|
|             Baja|16242576|
+-----------------+--------+



### Variable: Volumen de Transacciones

Se calculan percentiles sobre la variable de volumen de transacciones para clasificar los registros en niveles relativos de volumen de operación: Bajo, Medio y Alto.

In [13]:
# Calcular los percentiles para la columna 'Volume'
# Usamos approx_percentile para DataFrames grandes
volume_percentiles = df_partitioned.approxQuantile("Volume", [0.33, 0.66], 0.01)
p33 = volume_percentiles[0]
p66 = volume_percentiles[1]

print(f"Percentil 33 de Volumen: {p33:,.2f}")
print(f"Percentil 66 de Volumen: {p66:,.2f}")

df_partitioned = df_partitioned.withColumn("Volumen_Transacciones",
                                           when(col("Volume") <= p33, lit("Bajo"))
                                           .when((col("Volume") > p33) & (col("Volume") <= p66), lit("Medio"))
                                           .otherwise(lit("Alto")))

# Mostrar la distribución de los volúmenes de transacciones
df_partitioned.groupBy("Volumen_Transacciones").count().show()

Percentil 33 de Volumen: 9,200.00
Percentil 66 de Volumen: 200,100.00
+---------------------+--------+
|Volumen_Transacciones|   count|
+---------------------+--------+
|                Medio|11418009|
|                 Alto|11869581|
|                 Bajo|11358668|
+---------------------+--------+



### Combinaciones de estratos generadas

A partir de las tres variables de segmentación seleccionadas se generan combinaciones de estratos para representar subconjuntos homogéneos del dataset.

Dado que cada variable posee tres categorías, el número máximo teórico de combinaciones posibles es: 3 × 3 × 3 = 27 estratos.

Sin embargo, el número real dependerá de las combinaciones efectivamente presentes en los datos históricos analizados.

In [14]:
total_registros = df_partitioned.count()

df_estratos = df_partitioned.groupBy(
    "Periodo_Economico",
    "Nivel_Volatilidad",
    "Volumen_Transacciones"
).count()

df_estratos = df_estratos.withColumn(
    "Porcentaje",
    round((col("count") / lit(total_registros)) * 100, 4)
)

df_estratos.orderBy(col("Porcentaje").desc()).show(30, truncate=False)

+-----------------+-----------------+---------------------+-------+----------+
|Periodo_Economico|Nivel_Volatilidad|Volumen_Transacciones|count  |Porcentaje|
+-----------------+-----------------+---------------------+-------+----------+
|Recuperación     |Baja             |Bajo                 |6621937|19.113    |
|Recuperación     |Media            |Alto                 |4390511|12.6724   |
|Recuperación     |Baja             |Medio                |3841333|11.0873   |
|Recuperación     |Baja             |Alto                 |3473234|10.0248   |
|Recuperación     |Media            |Medio                |3354626|9.6825    |
|Recuperación     |Alta             |Medio                |2256682|6.5135    |
|Recuperación     |Alta             |Alto                 |2025171|5.8453    |
|Recuperación     |Alta             |Bajo                 |1656268|4.7805    |
|Recuperación     |Media            |Bajo                 |1259405|3.635     |
|Crisis           |Baja             |Bajo           

### Combinación de Particiones
Ahora, podemos ver cómo se combinan estas variables para formar las particiones. Mostraremos los primeros registros del DataFrame con las nuevas columnas de particionamiento.

In [15]:
df_partitioned.select("Date", "Ticker", "Volumen_Transacciones", "Nivel_Volatilidad", "Periodo_Economico").show(10)

# Opcionalmente, puedes contar las ocurrencias de una combinación de particiones específica
df_partitioned.groupBy("Periodo_Economico", "Nivel_Volatilidad", "Volumen_Transacciones").count().show(10, truncate=False)

+----------+------+---------------------+-----------------+-----------------+
|      Date|Ticker|Volumen_Transacciones|Nivel_Volatilidad|Periodo_Economico|
+----------+------+---------------------+-----------------+-----------------+
|1962-01-02|    ED|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   CVX|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    GD|                 Alto|            Media|       Pre-crisis|
|1962-01-02|    BP|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   MSI|                Medio|            Media|       Pre-crisis|
|1962-01-02|   HON|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    FL|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    GT|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   JNJ|                 Bajo|             Baja|       Pre-crisis|
|1962-01-02|   MMM|                 Alto|            Media|     

### Distribución porcentual de los estratos

Se calcula la proporción de ocurrencia de cada combinación generada para utilizarla como base del muestreo estratificado proporcional.

In [16]:
from pyspark.sql.functions import count, lit, round

total_registros = df_partitioned.count()

df_estratos = df_partitioned.groupBy(
    "Periodo_Economico",
    "Nivel_Volatilidad",
    "Volumen_Transacciones"
).count()

df_estratos = df_estratos.withColumn(
    "Porcentaje",
    round((col("count") / lit(total_registros)) * 100, 4)
)

df_estratos.orderBy(col("Porcentaje").desc()).show(27, truncate=False)

+-----------------+-----------------+---------------------+-------+----------+
|Periodo_Economico|Nivel_Volatilidad|Volumen_Transacciones|count  |Porcentaje|
+-----------------+-----------------+---------------------+-------+----------+
|Recuperación     |Baja             |Bajo                 |6621937|19.113    |
|Recuperación     |Media            |Alto                 |4390511|12.6724   |
|Recuperación     |Baja             |Medio                |3841333|11.0873   |
|Recuperación     |Baja             |Alto                 |3473234|10.0248   |
|Recuperación     |Media            |Medio                |3354626|9.6825    |
|Recuperación     |Alta             |Medio                |2256682|6.5135    |
|Recuperación     |Alta             |Alto                 |2025171|5.8453    |
|Recuperación     |Alta             |Bajo                 |1656268|4.7805    |
|Recuperación     |Media            |Bajo                 |1259405|3.635     |
|Crisis           |Baja             |Bajo           

## 1 Construcción de la muestra M

A partir de las particiones (estratos) definidas con las tres variables de caracterización (Periodo económico, Nivel de volatilidad y Volumen de transacciones) se construye la muestra M. El objetivo es determinar cuántas instancias debe aportar cada partición, de tal forma que **M no introduzca ningún sesgo** que pueda alterar la calidad de los resultados de los modelos.

### Identificador de estrato
Cada combinación de las tres variables define un estrato de la población. Se concatena en una sola columna `Estrato`, que servirá como clave para el muestreo por partición.

### Criterio para evitar sesgo: asignación proporcional
En el método de Asignación proporcional cada estrato aporta un número de instancias proporcional a su peso en la población, constante para todos los estratos.

La asignación proporcional reproduce exactamente la composición de la población dentro de la muestra, por lo que es la opción que no introduce sesgo de representación.

### Tamaño de muestra (criterio estadístico, no arbitrario)
El tamaño total `n` se fija con la "fórmula de Cochran" con corrección por población finita, de modo que la representatividad de la muestra quede garantizada estadísticamente y no dependa de una elección arbitraria.

Con un nivel de confianza del 95 % (`Z = 1.96`), la varianza máxima (`p = 0.5`) y un margen de error del 1 % (`e = 0.01`). Ese valor constituye el **mínimo** que cualquier muestra representativa debe superar; la fracción proporcional adoptada se valida comprobando que `|M|` queda muy por encima de dicho mínimo.

In [17]:
df_partitioned = df_partitioned.withColumn(
    "Estrato",
    concat_ws(
        "_",
        col("Periodo_Economico"),
        col("Nivel_Volatilidad"),
        col("Volumen_Transacciones")
    )
)

df_partitioned.groupBy("Estrato").count().orderBy("count", ascending=False).show(30, truncate=False)

+---------------------------+-------+
|Estrato                    |count  |
+---------------------------+-------+
|Recuperación_Baja_Bajo     |6621937|
|Recuperación_Media_Alto    |4390511|
|Recuperación_Baja_Medio    |3841333|
|Recuperación_Baja_Alto     |3473234|
|Recuperación_Media_Medio   |3354626|
|Recuperación_Alta_Medio    |2256682|
|Recuperación_Alta_Alto     |2025171|
|Recuperación_Alta_Bajo     |1656268|
|Recuperación_Media_Bajo    |1259405|
|Crisis_Baja_Bajo           |893429 |
|Crisis_Alta_Alto           |658391 |
|Crisis_Media_Alto          |631243 |
|Crisis_Alta_Medio          |542005 |
|Crisis_Media_Medio         |457087 |
|Crisis_Baja_Medio          |357435 |
|Pre-crisis_Baja_Medio      |316259 |
|Crisis_Alta_Bajo           |295893 |
|Pre-crisis_Baja_Bajo       |286782 |
|Pre-crisis_Baja_Alto       |245539 |
|Pre-crisis_Media_Medio     |223470 |
|Crisis_Baja_Alto           |206628 |
|Pre-crisis_Media_Alto      |201031 |
|Crisis_Media_Bajo          |163874 |
|Pre-crisis_

In [20]:
# --- Tamaño de la población particionada (todas las instancias de D con su estrato asignado) ---
N = df_partitioned.count()

# --- (1) Tamaño de muestra MÍNIMO mediante la fórmula de Cochran ---
# Criterio estadístico (no arbitrario) que garantiza la representatividad de la muestra.
Z, p, e = 1.96, 0.5, 0.01                  # 95% de confianza, varianza máxima, error del 1%
n0 = (Z ** 2 * p * (1 - p)) / (e ** 2)     # tamaño sin corregir
n_min = n0 / (1 + (n0 - 1) / N)            # corrección por población finita
print(f"Población N ...............................: {N:,}")
print(f"Tamaño mínimo de muestra (Cochran 95%, e=1%): {n_min:,.0f}")

# --- (2) Fracción de muestreo PROPORCIONAL adoptada ---
# La misma fracción f en todos los estratos => asignación proporcional => sin sesgo.
# Se adopta f = 5%, que produce un |M| muy por encima del mínimo de Cochran, dejando
# datos abundantes para los algoritmos de aprendizaje sin distorsionar la distribución real.
f = 0.05
n_obj = int(N * f)
print(f"Fracción proporcional adoptada f ..........: {f:.2%}")
print(f"Tamaño objetivo de la muestra |M| .........: ~{n_obj:,}")
print(f"|M| objetivo supera el mínimo de Cochran ..: {n_obj > n_min}\n")

# --- (3) Número de instancias que debe aportar CADA partición: n_h = round(f * N_h) ---
# Tamaño de cada estrato (N_h) en la población particionada.
conteos_estrato = (df_partitioned.groupBy("Estrato").count()
                   .withColumnRenamed("count", "N_h"))

# Asignación proporcional: peso del estrato, instancias objetivo y fracción (constante).
asignacion = (conteos_estrato
    .withColumn("Peso_W_h_%",   round(col("N_h") / lit(N) * 100, 4))      # peso del estrato en D
    .withColumn("n_h_objetivo", round(col("N_h") * lit(f)).cast("long"))  # instancias a muestrear
    .withColumn("fraccion_f_h", lit(f))                                   # fracción (constante)
    .orderBy(col("N_h").desc()))

print("Número de instancias a muestrear por partición (asignación proporcional):")
asignacion.show(30, truncate=False)

Población N ...............................: 34,646,258
Tamaño mínimo de muestra (Cochran 95%, e=1%): 9,601
Fracción proporcional adoptada f ..........: 5.00%
Tamaño objetivo de la muestra |M| .........: ~1,732,312
|M| objetivo supera el mínimo de Cochran ..: True

Número de instancias a muestrear por partición (asignación proporcional):
+---------------------------+-------+----------+------------+------------+
|Estrato                    |N_h    |Peso_W_h_%|n_h_objetivo|fraccion_f_h|
+---------------------------+-------+----------+------------+------------+
|Recuperación_Baja_Bajo     |6621937|19.113    |331097      |0.05        |
|Recuperación_Media_Alto    |4390511|12.6724   |219526      |0.05        |
|Recuperación_Baja_Medio    |3841333|11.0873   |192067      |0.05        |
|Recuperación_Baja_Alto     |3473234|10.0248   |173662      |0.05        |
|Recuperación_Media_Medio   |3354626|9.6825    |167731      |0.05        |
|Recuperación_Alta_Medio    |2256682|6.5135    |112834      

In [21]:
# --- Generación de la muestra M por particiones (PySpark) ---
# sampleBy aplica un muestreo de Bernoulli por estrato con la fracción indicada.
# Al usar la MISMA fracción f en todos los estratos, la asignación es proporcional (sin sesgo).
fracciones = {r["Estrato"]: f for r in conteos_estrato.collect()}
sampled_df = df_partitioned.sampleBy("Estrato", fractions=fracciones, seed=42).cache()

# Tamaño de la muestra M resultante frente al objetivo proporcional
n_M = sampled_df.count()
print(f"Total población particionada : {N:,}")
print(f"Total muestra M obtenida ... : {n_M:,}")
print(f"Tamaño objetivo |M| ........ : ~{n_obj:,}\n")

# --- Verificación de AUSENCIA DE SESGO ---
# La distribución de estratos en M debe coincidir con la de la población D:
# si los pesos por estrato son prácticamente idénticos, no se ha inyectado sesgo.
# Nota: Spark resuelve los nombres de columna SIN distinguir mayúsculas/minúsculas, por lo que
# los conteos de población y de muestra se nombran "N_pob" y "n_muestra" (no "N_h"/"n_h",
# que Spark consideraría la misma columna y provocaría una ambigüedad al unir).
dist_pob = (df_partitioned.groupBy("Estrato").count().withColumnRenamed("count", "N_pob")
            .withColumn("W_pob_%", round(col("N_pob") / lit(N) * 100, 3)))
dist_M = (sampled_df.groupBy("Estrato").count().withColumnRenamed("count", "n_muestra")
          .withColumn("W_M_%", round(col("n_muestra") / lit(n_M) * 100, 3)))

comparacion = (dist_pob.join(dist_M, "Estrato")
    .withColumn("dif_%", round(col("W_M_%") - col("W_pob_%"), 3))   # ~0 => sin sesgo
    .orderBy(col("N_pob").desc()))

print("Comparación de pesos por estrato (población D vs. muestra M):")
comparacion.show(30, truncate=False)

Total población particionada : 34,646,258
Total muestra M obtenida ... : 1,732,359
Tamaño objetivo |M| ........ : ~1,732,312

Comparación de pesos por estrato (población D vs. muestra M):
+---------------------------+-------+-------+---------+------+------+
|Estrato                    |N_pob  |W_pob_%|n_muestra|W_M_% |dif_% |
+---------------------------+-------+-------+---------+------+------+
|Recuperación_Baja_Bajo     |6621937|19.113 |330629   |19.085|-0.028|
|Recuperación_Media_Alto    |4390511|12.672 |219466   |12.669|-0.003|
|Recuperación_Baja_Medio    |3841333|11.087 |191471   |11.053|-0.034|
|Recuperación_Baja_Alto     |3473234|10.025 |174095   |10.05 |0.025 |
|Recuperación_Media_Medio   |3354626|9.683  |167555   |9.672 |-0.011|
|Recuperación_Alta_Medio    |2256682|6.513  |113287   |6.539 |0.026 |
|Recuperación_Alta_Alto     |2025171|5.845  |101285   |5.847 |0.002 |
|Recuperación_Alta_Bajo     |1656268|4.781  |82543    |4.765 |-0.016|
|Recuperación_Media_Bajo    |1259405|3.635

### Preparación de los datos

Sobre la muestra estratificada **M** (`sampled_df`) obtenida en el paso anterior se aplican estrategias de corrección para dejar un conjunto **listo para los algoritmos de aprendizaje**. El pre-procesamiento contempla tres tareas:

1. **Corrección de valores nulos:** diagnóstico y eliminación de registros con nulos en columnas críticas.
2. **Identificación de valores atípicos:** detección y eliminación de registros corruptos (precios negativos o magnitudes imposibles).
3. **Transformación de tipos de datos:** conversión de tipos y generación de una variable derivada (`retorno_pct`).

El resultado es una muestra **M pre-procesada** consistente y depurada.

In [22]:
# Trabajamos sobre la muestra estratificada M obtenida en el paso de selección
M = sampled_df
M.cache()
total_M = M.count()
print(f"Registros en la muestra M: {total_M:,}\n")

# Diagnóstico de valores nulos por columna
nulos_M = M.select([count(when(col(c).isNull(), 1)).alias(c) for c in M.columns]).toPandas().T
nulos_M.columns = ["Valores nulos"]
nulos_M["%"] = (nulos_M["Valores nulos"] / total_M) * 100
nulos_M

Registros en la muestra M: 1,732,359



,Valores nulos,%
Date,0,0.000000
Ticker,0,0.000000
Open,4,0.000231
High,4,0.000231
Low,4,0.000231
Close,4,0.000231
Volume,0,0.000000
Dividends,0,0.000000
Stock Splits,0,0.000000
Periodo_Economico,0,0.000000


In [23]:
# Eliminación de registros con nulos en las columnas críticas (precios y volumen).
#     Representan < 0.001% de la muestra, por lo que su eliminación no introduce sesgo.
cols_criticas = ["Open", "High", "Low", "Close", "Volume"]
M_clean = M.dropna(subset=cols_criticas)

# Eliminación de registros 'Sin dato' en volatilidad (Low = 0 => división indefinida)
M_clean = M_clean.filter(col("Nivel_Volatilidad") != "Sin dato")

print(f"Registros tras tratar nulos : {M_clean.count():,}")
print(f"Registros eliminados        : {M.count() - M_clean.count():,}")

Registros tras tratar nulos : 1,732,307
Registros eliminados        : 52


### Identificación de valores atípicos

El resumen estadístico inicial reveló valores **imposibles**: precios negativos y magnitudes del orden de 10²³–10²⁸, producto de errores de captura en la fuente. Se aplican dos correcciones:

- **Consistencia física:** precios no negativos y `High ≥ Low` (un máximo nunca puede ser menor que el mínimo del día).
- **Tope físico sobre los precios:** la distribución de precios es **bimodal** —valores legítimos por debajo de ~10⁶ USD y registros corruptos del orden de 10²³–10²⁸, *sin valores intermedios*—. Por esa razón un **recorte por percentiles no funciona**: el percentil 99.9% cae dentro de la zona corrupta y no descarta nada. Tomando como referencia el precio por acción más alto de la historia (~700,000 USD, Berkshire Hathaway), se fija un umbral seguro en **1,000,000 USD**, que elimina únicamente los registros imposibles y conserva todas las acciones legítimas. El volumen no se filtra, pues sus valores máximos (~10⁹) son plausibles para días de alta liquidez.

In [24]:
# (a) Filtros de consistencia física: precios no negativos y High >= Low
M_clean = M_clean.filter(
    (col("Open") >= 0) & (col("High") >= 0) &
    (col("Low") >= 0) & (col("Close") >= 0) &
    (col("High") >= col("Low"))
)

# (b) Eliminación de registros corruptos mediante un TOPE FÍSICO sobre los precios.
#     La distribución de precios es BIMODAL: valores legítimos (< ~10^6 USD) y registros
#     corruptos del orden de 10^23 - 10^28, SIN valores intermedios. Con esa forma, un recorte
#     por percentiles NO funciona (el percentil 99.9% cae dentro de la zona corrupta y no filtra
#     nada). Tomando como referencia el precio por acción más alto de la historia (~700,000 USD,
#     Berkshire Hathaway), se fija un umbral físico seguro en 1,000,000 USD: descarta únicamente
#     los valores imposibles y conserva toda acción legítima.
TOPE_PRECIO = 1_000_000.0
n_antes = M_clean.count()
for c in ["Open", "High", "Low", "Close"]:
    M_clean = M_clean.filter(col(c) <= TOPE_PRECIO)

n_despues = M_clean.count()
print(f"Registros corruptos eliminados  : {n_antes - n_despues:,}")
print(f"Registros tras eliminar atípicos: {n_despues:,}\n")

# Verificación: el resumen ya no presenta magnitudes imposibles
M_clean.select("Open", "High", "Low", "Close", "Volume") \
       .describe().toPandas().set_index("summary").T

Registros corruptos eliminados  : 6,841
Registros tras eliminar atípicos: 1,720,913



summary,count,mean,stddev,min,max
Open,1720913,1360.5654674605032,23230.425080868186,0.0,1000000.0
High,1720913,1425.596651408109,24126.540482104123,1.0249949500273914E-10,1000000.0
Low,1720913,1335.5765823269758,22711.38706636596,1.0249949500273914E-10,1000000.0
Close,1720913,1378.8734264490074,23375.03674859062,1.0249949500273914E-10,1000000.0
Volume,1720913,1340778.090313107,1.4765324701510247E7,0.0,3.7554384E9


In [25]:
# Transformación de tipos de datos y generación de variable derivada
from pyspark.sql.functions import expr

M_prep = (M_clean
    # Dividends y Stock Splits venían como Decimal -> se convierten a double (requerido por MLlib)
    .withColumn("Dividends", col("Dividends").cast("double"))
    .withColumn("Stock Splits", col("Stock Splits").cast("double"))
    # Retorno intradía (%) = (Close - Open)/Open ; try_divide evita la división por cero
    .withColumn("retorno_pct", expr("try_divide((Close - Open) * 100, Open)"))
    .fillna({"retorno_pct": 0.0})
)
M_prep.cache()
print(f"Muestra M pre-procesada lista: {M_prep.count():,} registros\n")

# Verificación: el resumen ya no presenta valores imposibles (negativos ni magnitudes 10^25)
M_prep.select("Open", "High", "Low", "Close", "Volume", "retorno_pct") \
      .describe().toPandas().set_index("summary").T

Muestra M pre-procesada lista: 1,720,913 registros



summary,count,mean,stddev,min,max
Open,1720913,1360.5654674605032,23230.425080868186,0.0,1000000.0
High,1720913,1425.596651408109,24126.540482104123,1.0249949500273914E-10,1000000.0
Low,1720913,1335.5765823269758,22711.38706636596,1.0249949500273914E-10,1000000.0
Close,1720913,1378.8734264490074,23375.03674859062,1.0249949500273914E-10,1000000.0
Volume,1720913,1340778.090313107,1.4765324701510247E7,0.0,3.7554384E9
retorno_pct,1720913,0.386243413318427,105.35985505118214,-99.62162163998956,129903.46888420406


## 2 Construcción Train – Test

Una vez construida y depurada la muestra M, se divide en un conjunto de entrenamiento (Tr) y uno de prueba (Ts). La partición debe cumplir dos propiedades formales:

- **Disyunción:** ningún registro puede estar a la vez en entrenamiento y prueba (evita fuga de información y métricas infladas).
- **Cobertura total:** la unión de todas las particiones reconstruye exactamente M (no se pierde ni se duplica ningún registro).

### Cálculo del porcentaje de división
Se adopta una división 80 % entrenamiento / 20 % prueba, aplicada **dentro de cada estrato** $M_i$. La justificación:

- El porcentaje no se aplica de forma global, sino por estrato (muestreo estratificado, retomando la estrategia del paso anterior). Así, la proporción de cada patrón (combinación Periodo × Volatilidad × Volumen) se conserva idéntica en Tr y en Ts, de modo que no se desvía la probabilidad de ocurrencia de los patrones en ninguna de las dos particiones. Un muestreo aleatorio simple podría, por azar, sub-representar un estrato minoritario en prueba e inyectar sesgo.
- Con una muestra de ~1.7 M registros, el 20 % de prueba equivale a ~345 mil instancias: una muestra de prueba lo bastante grande para estimar las métricas con varianza muy baja (intervalos de confianza estrechos), mientras el 80 % de entrenamiento** aporta datos abundantes para que los modelos aprendan patrones estables.
- Reservar más para prueba (p. ej. 30 %) no mejora la precisión de las métricas y resta datos de aprendizaje. Además, 80/20 es el estándar más extendido en ML.

### Garantía de las propiedades
La disyunción se garantiza tomando el conjunto de prueba como el **complemento exacto** del de entrenamiento mediante un *anti-join* sobre un identificador único: lo que no cae en Tr cae en Ts, sin solapamiento ni pérdida. Esto asegura simultáneamente $Tr \cap Ts = \varnothing$ y $Tr \cup Ts = M$. Las tres propiedades (disyunción, cobertura y ausencia de sesgo) se comprueban explícitamente más abajo.

In [26]:
from pyspark.sql.functions import monotonically_increasing_id, round as _round

# --- Porcentaje de división (se aplica POR ESTRATO, no de forma global) ---
P_TRAIN = 0.8            # 80% entrenamiento
P_TEST  = 1 - P_TRAIN    # 20% prueba
print(f"Porcentaje de división: {P_TRAIN:.0%} entrenamiento / {P_TEST:.0%} prueba\n")

# Identificador único por registro para particionar SIN solapamiento.
# Se materializa con cache()+count() para que 'id' sea estable (monotonically_increasing_id
# es no determinista si el DataFrame se recalcula).
M_model = M_prep.withColumn("id", monotonically_increasing_id()).cache()
n_M_model = M_model.count()

# --- Muestreo ESTRATIFICADO: se toma P_TRAIN de CADA estrato (Mi) para entrenamiento (Tri) ---
estratos_M = [r["Estrato"] for r in M_model.select("Estrato").distinct().collect()]
fracciones_train = {e: P_TRAIN for e in estratos_M}
train_df = M_model.sampleBy("Estrato", fractions=fracciones_train, seed=42)

# --- El conjunto de prueba (Tsi) es el COMPLEMENTO exacto (anti-join por id) ---
# Tomar el complemento garantiza simultáneamente  Tr ∩ Ts = ∅  y  Tr ∪ Ts = M.
test_df = M_model.join(train_df.select("id"), on="id", how="left_anti")

train_df.cache(); test_df.cache()
n_train, n_test = train_df.count(), test_df.count()
print(f"Conjunto de entrenamiento (Tr): {n_train:,} ({n_train / n_M_model * 100:.1f}%)")
print(f"Conjunto de prueba        (Ts): {n_test:,} ({n_test / n_M_model * 100:.1f}%)")

Porcentaje de división: 80% entrenamiento / 20% prueba

Conjunto de entrenamiento (Tr): 1,376,087 (80.0%)
Conjunto de prueba        (Ts): 344,826 (20.0%)


In [27]:
# --- Verificación de las propiedades formales de la partición Train/Test ---

# (1) Disyunción:  Tr ∩ Ts = ∅   (la intersección de ids debe ser 0)
interseccion = train_df.select("id").intersect(test_df.select("id")).count()
print(f"(1) Disjunción   |Tr ∩ Ts| = {interseccion}  ->  {'vacío (disjuntos)' if interseccion == 0 else 'HAY SOLAPAMIENTO'}")

# (2) Cobertura total:  |Tr| + |Ts| == |M|   (Tr ∪ Ts reconstruye M, sin pérdidas ni duplicados)
union_total = n_train + n_test
print(f"(2) Cobertura    |Tr| + |Ts| = {union_total:,}  ==  |M| = {n_M_model:,}  ->  {union_total == n_M_model}")

# (3) Ausencia de sesgo: la proporción ~80/20 se conserva en CADA estrato,
#     por lo que la probabilidad de ocurrencia de cada patrón no se desvía en Tr ni en Ts.
dist = (train_df.groupBy("Estrato").count().withColumnRenamed("count", "Tr")
        .join(test_df.groupBy("Estrato").count().withColumnRenamed("count", "Ts"), "Estrato")
        .withColumn("%Tr", _round(col("Tr") / (col("Tr") + col("Ts")) * 100, 1))
        .withColumn("%Ts", _round(col("Ts") / (col("Tr") + col("Ts")) * 100, 1))
        .orderBy(col("Tr").desc()))

print("\n(3) Proporción por estrato (debe rondar 80/20 en todos los estratos -> sin sesgo):")
dist.show(30, truncate=False)

(1) Disjunción   |Tr ∩ Ts| = 0  ->  vacío (disjuntos)
(2) Cobertura    |Tr| + |Ts| = 1,720,913  ==  |M| = 1,720,913  ->  True

(3) Proporción por estrato (debe rondar 80/20 en todos los estratos -> sin sesgo):
+------------------------+------+-----+----+----+
|Estrato                 |Tr    |Ts   |%Tr |%Ts |
+------------------------+------+-----+----+----+
|Recuperación_Baja_Bajo  |260489|65075|80.0|20.0|
|Recuperación_Media_Alto |175387|43957|80.0|20.0|
|Recuperación_Baja_Medio |152776|38391|79.9|20.1|
|Recuperación_Baja_Alto  |139088|34722|80.0|20.0|
|Recuperación_Media_Medio|133647|33758|79.8|20.2|
|Recuperación_Alta_Medio |90423 |22806|79.9|20.1|
|Recuperación_Alta_Alto  |81027 |20238|80.0|20.0|
|Recuperación_Alta_Bajo  |63773 |15841|80.1|19.9|
|Recuperación_Media_Bajo |49431 |12483|79.8|20.2|
|Crisis_Baja_Bajo        |35399 |8935 |79.8|20.2|
|Crisis_Alta_Alto        |26557 |6549 |80.2|19.8|
|Crisis_Media_Alto       |25276 |6339 |79.9|20.1|
|Crisis_Alta_Medio       |21966 |5414 |8

## 3 Selección de métricas para medir calidad de resultados

En la etapa de entrenamiento se construyen dos tipos de modelos —uno **supervisado** (árbol de decisión, clasificación multiclase del `Nivel_Volatilidad`) y uno **no supervisado** (K-Means)—, por lo que la calidad de sus resultados se mide con familias de métricas distintas. Además, al trabajar con **grandes volúmenes de datos** se impone un criterio transversal: las métricas elegidas deben ser **escalables**, es decir, calculables mediante agregaciones distribuidas en pocas pasadas y sin operaciones de coste cuadrático $O(n^2)$ ni necesidad de materializar todo el conjunto en memoria del driver.

### Criterio para grandes volúmenes de datos
- **Escalabilidad / coste lineal:** se priorizan métricas obtenibles con agregaciones distribuidas (conteos, sumas) y se evita cualquier cálculo par-a-par $O(n^2)$.
- **Evaluadores nativos de Spark MLlib:** `MulticlassClassificationEvaluator` y `ClusteringEvaluator` calculan las métricas de forma distribuida, sin recolectar los datos en el driver.
- **Estimación de baja varianza:** el conjunto de prueba (~345 mil registros) es lo bastante grande para que las métricas estimadas tengan intervalos de confianza muy estrechos.

### a) Modelo supervisado — clasificación multiclase
| Métrica | Qué mide | Por qué (con desbalance / big data) |
|---|---|---|
| **Exactitud (accuracy)** | Porcentaje de predicciones correctas | Intuitiva y barata, pero **sensible al desbalance** de estratos; insuficiente por sí sola. |
| **F1 ponderado** | Media armónica de precisión y recall, ponderada por soporte | Métrica **principal**: equilibra precisión/recall y refleja el desbalance real de clases. |
| **Precisión y recall (por clase)** | Errores de comisión / omisión en cada clase | Revela si una clase (p. ej. *Alta*) se predice peor que las demás. |
| **Matriz de confusión** | Distribución de aciertos y confusiones entre clases | Apoyo **diagnóstico**: muestra si los errores se concentran entre clases adyacentes. |

**Selección argumentada:** se usarán **exactitud + F1 ponderado** como métricas cuantitativas principales (el F1 por su robustez ante el desbalance de estratos), apoyadas en la **matriz de confusión** como herramienta de diagnóstico cualitativo. Todas se obtienen con `MulticlassClassificationEvaluator`, escalable a grandes volúmenes.

### b) Modelo no supervisado — agrupamiento (K-Means)
| Métrica | Qué mide | Por qué (con big data) |
|---|---|---|
| **Coeficiente de silueta** | Cohesión intra-clúster frente a separación inter-clúster, en $[-1, 1]$ | Métrica **principal** de calidad del agrupamiento. Spark usa la **silueta con distancia euclidiana al cuadrado**, una aproximación **distribuible** que evita el coste $O(n^2)$ de la silueta clásica. |
| **Costo / WSSSE (inercia)** | Suma de distancias al cuadrado dentro de cada clúster | Base del **método del codo** para elegir `k`; se calcula de forma distribuida. |
| **Tamaño y balance de los clústeres** | Número de instancias por grupo | Detecta soluciones **degeneradas** (clústeres vacíos o dominados por outliers), no evidentes solo con la silueta. |

**Selección argumentada:** se usará el **coeficiente de silueta** junto con el **costo (WSSSE / método del codo)** para seleccionar el número de clústeres `k` y evaluar la calidad, complementados con la **inspección del tamaño de los clústeres** para descartar agrupamientos degenerados. Estas métricas están disponibles en `ClusteringEvaluator` y `KMeansSummary`, escalables a grandes volúmenes.

### Resumen de métricas seleccionadas
- **Supervisado:** exactitud, **F1 ponderado** (principal) y matriz de confusión.
- **No supervisado:** **coeficiente de silueta** (principal), costo / WSSSE (codo) y tamaño de los clústeres.
- Todas son **calculables de forma distribuida** en PySpark, requisito indispensable dado el volumen de datos.

## 4 Entrenamiento de Modelos de Aprendizaje

En esta etapa se entrenan los dos modelos del trabajo (un árbol de decisión supervisado y un K-Means no supervisado) siguiendo una estrategia de entrenamiento explícita, cuyo objetivo es construir modelos capaces de identificar los patrones de interés en los datos sin quedar sobre-ajustados.

La estrategia común consta de:

- Procesamiento de los datos dentro de un Pipeline de Spark MLlib, de modo que las transformaciones (indexado, ensamblado, escalado) se ajusten únicamente con el conjunto de entrenamiento y se apliquen de forma idéntica al de prueba. Así se evita la fuga de información.
- Uso exclusivo de `train_df` para ajustar parámetros e hiper-parámetros; `test_df` se reserva intacto para estimar la generalización.
- Ajuste de hiper-parámetros mediante búsqueda en rejilla con validación, eligiendo la configuración que maximiza la métrica seleccionada en la sección 3.
- Control del sobreajuste con técnicas propias de cada algoritmo (pre-poda y límites de complejidad en el árbol; estandarización, selección prudente de k y descarte de soluciones degeneradas en K-Means) y con la comparación entre el desempeño en entrenamiento y en prueba.

### 4.1 Modelo supervisado: árbol de decisión

Procesamiento: la etiqueta `Nivel_Volatilidad` y la variable categórica `Periodo_Economico` se convierten a índices numéricos; el resto de predictores se ensamblan en un vector. Se excluyen `High`, `Low` y `Volatilidad_Diaria_Pct` porque la etiqueta se derivó de ellas y su inclusión provocaría fuga de información.

Ajuste de hiper-parámetros: se explora una rejilla sobre `maxDepth`, `minInstancesPerNode` y `maxBins` con `TrainValidationSplit`. Se prefiere a la validación cruzada de k pliegues porque, con grandes volúmenes de datos, una única partición de validación es mucho más económica y, por la abundancia de datos, igualmente fiable.

Prevención del sobreajuste: se limita la profundidad del árbol (`maxDepth`) y se exige un mínimo de instancias por hoja (`minInstancesPerNode`, pre-poda); además se contrasta el F1 de entrenamiento contra el de prueba para confirmar que el modelo generaliza y no memoriza.

In [28]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1) Procesamiento dentro del Pipeline (se ajusta solo con el train y se reaplica al test).
#    Etiqueta categórica -> índice numérico
label_indexer = StringIndexer(inputCol="Nivel_Volatilidad", outputCol="label", handleInvalid="skip")
#    Variable categórica predictora -> índice numérico
periodo_indexer = StringIndexer(inputCol="Periodo_Economico", outputCol="Periodo_idx", handleInvalid="keep")

# 2) Variables predictoras. Se EXCLUYEN High, Low y Volatilidad_Diaria_Pct: la etiqueta
#    Nivel_Volatilidad se derivó de ellas y usarlas sería fuga de información.
feature_cols = ["Open", "Close", "Volume", "Dividends", "Stock Splits", "retorno_pct", "Periodo_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

# 3) Algoritmo base (el mismo del trabajo previo): árbol de decisión.
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", seed=42)
pipeline_dt = Pipeline(stages=[label_indexer, periodo_indexer, assembler, dt])

# 4) Rejilla de hiper-parámetros. Los parámetros elegidos controlan el sobreajuste:
#    - maxDepth: árboles menos profundos generalizan mejor (limita la complejidad).
#    - minInstancesPerNode: pre-poda; exige un mínimo de instancias por hoja.
#    - maxBins: número de cortes al discretizar variables continuas.
grid_dt = (ParamGridBuilder()
           .addGrid(dt.maxDepth, [6, 10])
           .addGrid(dt.minInstancesPerNode, [50, 500])
           .addGrid(dt.maxBins, [32])
           .build())

# 5) Métrica de selección: F1 ponderado (robusto ante el desbalance de clases; ver sección 3).
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                                 metricName="f1")

# 6) Ajuste de hiper-parámetros con TrainValidationSplit (partición interna 80/20 del train).
#    La validación interna es lo que impide elegir una configuración sobre-ajustada.
tvs_dt = TrainValidationSplit(estimator=pipeline_dt,
                              estimatorParamMaps=grid_dt,
                              evaluator=evaluator_f1,
                              trainRatio=0.8,
                              parallelism=2,
                              seed=42)

# 7) Entrenamiento (solo con train_df; test_df se reserva para la evaluación final).
modelo_dt_tvs = tvs_dt.fit(train_df)
modelo_dt = modelo_dt_tvs.bestModel          # mejor pipeline según la validación

# Hiper-parámetros seleccionados
mejor_dt = modelo_dt.stages[-1]
print("Hiper-parámetros elegidos para el árbol de decisión:")
print(f"  maxDepth            = {mejor_dt.getMaxDepth()}")
print(f"  minInstancesPerNode = {mejor_dt.getMinInstancesPerNode()}")
print(f"  maxBins             = {mejor_dt.getMaxBins()}")

# 8) Diagnóstico de sobreajuste: F1 en entrenamiento vs. prueba.
#    Una diferencia pequeña indica que el modelo generaliza (no memoriza el train).
f1_train = evaluator_f1.evaluate(modelo_dt.transform(train_df))
f1_test = evaluator_f1.evaluate(modelo_dt.transform(test_df))
print(f"\nF1 en entrenamiento ......: {f1_train:.4f}")
print(f"F1 en prueba .............: {f1_test:.4f}")
print(f"Diferencia (train - test) : {f1_train - f1_test:.4f}  (cercana a 0 => sin sobreajuste)")

Hiper-parámetros elegidos para el árbol de decisión:
  maxDepth            = 10
  minInstancesPerNode = 50
  maxBins             = 32

F1 en entrenamiento ......: 0.7264
F1 en prueba .............: 0.7254
Diferencia (train - test) : 0.0010  (cercana a 0 => sin sobreajuste)


### 4.2 Modelo no supervisado: K-Means

Procesamiento: las variables se estandarizan (media 0, desviación 1) porque K-Means se basa en distancias euclidianas y, sin escalar, la variable de mayor magnitud dominaría la formación de los grupos. Como ajuste adicional se aplica una transformación logarítmica (`log1p`) a precios y volumen para comprimir su fuerte asimetría, que en el trabajo previo producía clústeres degenerados; de este modo el modelo puede descubrir patrones de interés en lugar de aislar unos pocos valores extremos.

Ajuste del hiper-parámetro k: se evalúan varios valores de k combinando el costo (método del codo) y el coeficiente de silueta. La selección de k es automática: se toma el k con mayor silueta entre las soluciones no degeneradas (aquellas cuyo clúster más pequeño supera el 1 % de los datos). Elegir un k excesivo es el análogo al sobreajuste, por lo que esta regla lo previene.

Evaluación: el preprocesamiento se ajusta solo con el train y la calidad se verifica también sobre el test, comprobando que la estructura de clústeres se mantiene.

In [29]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.functions import log1p

# 1) Procesamiento de datos para el agrupamiento.
#    Ajuste adicional: log1p sobre precios y volumen. Su fuerte ASIMETRÍA a la derecha hacía
#    que K-Means formara clústeres degenerados dominados por unos pocos valores extremos;
#    la transformación logarítmica comprime esa cola y revela los patrones de interés.
def preparar_km(df):
    out = df
    for c in ["Open", "High", "Low", "Close", "Volume"]:
        out = out.withColumn(c + "_log", log1p(col(c)))
    return out

features_km_log = ["Open_log", "High_log", "Low_log", "Close_log", "Volume_log",
                   "Volatilidad_Diaria_Pct"]

train_km_src = preparar_km(train_df)
test_km_src = preparar_km(test_df)

# Ensamblado + estandarización. El preprocesamiento se AJUSTA solo con el train (sin fuga de info).
assembler_km = VectorAssembler(inputCols=features_km_log, outputCol="features_raw",
                               handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled",
                        withMean=True, withStd=True)
prep_km = Pipeline(stages=[assembler_km, scaler]).fit(train_km_src)
train_km = prep_km.transform(train_km_src).cache()
test_km = prep_km.transform(test_km_src)

# 2) Ajuste del hiper-parámetro k por método del codo (costo) + coeficiente de silueta.
evaluator_km = ClusteringEvaluator(featuresCol="features_scaled", metricName="silhouette",
                                   distanceMeasure="squaredEuclidean")

print(f"{'k':>2} | {'silueta':>9} | {'costo (WSSSE)':>16} | tamaño de clústeres")
print("-" * 70)
resultados_k = {}
for k in range(2, 8):
    km = KMeans(featuresCol="features_scaled", k=k, seed=42, initSteps=5, maxIter=30)
    modelo = km.fit(train_km)
    sil = evaluator_km.evaluate(modelo.transform(train_km))
    costo = modelo.summary.trainingCost
    tam = modelo.summary.clusterSizes
    resultados_k[k] = (sil, costo, modelo)
    print(f"{k:>2} | {sil:>9.4f} | {costo:>16,.1f} | {tam}")

# 3) Selección automática de k: mayor silueta entre las soluciones NO degeneradas
#    (clúster más pequeño >= 1% de los datos). Previene un k excesivo (análogo al sobreajuste).
n_train_km = train_km.count()
umbral = 0.01 * n_train_km
no_degenerados = {k: sil for k, (sil, costo, modelo) in resultados_k.items()
                  if min(modelo.summary.clusterSizes) >= umbral}
if no_degenerados:
    K_OPT = max(no_degenerados, key=no_degenerados.get)
else:
    K_OPT = max(resultados_k, key=lambda k: resultados_k[k][0])
print(f"\nk seleccionado (mejor silueta sin clústeres degenerados): {K_OPT}")

modelo_km = resultados_k[K_OPT][2]

# 4) Evaluación de generalización sobre el conjunto de prueba (no usado al ajustar).
sil_test = evaluator_km.evaluate(modelo_km.transform(test_km))
print(f"Silueta en prueba (k={K_OPT}): {sil_test:.4f}")
print("Tamaño de cada clúster en prueba (verifica ausencia de grupos degenerados):")
modelo_km.transform(test_km).groupBy("prediction").count().orderBy("prediction").show()

 k |   silueta |    costo (WSSSE) | tamaño de clústeres
----------------------------------------------------------------------
 2 |    0.5044 |      5,349,091.1 | [588515, 787572]
 3 |    0.5168 |      3,993,858.8 | [733424, 591573, 51090]
 4 |    0.5862 |      2,695,490.1 | [735621, 588198, 52267, 1]
 5 |    0.4058 |      3,071,361.1 | [531573, 118022, 42004, 547414, 137074]
 6 |    0.5573 |      1,590,763.7 | [367056, 29013, 157499, 240968, 581550, 1]
 7 |    0.5712 |      1,308,311.1 | [358960, 99009, 231488, 544606, 111135, 1, 30888]

k seleccionado (mejor silueta sin clústeres degenerados): 3
Silueta en prueba (k=3): 0.5886
Tamaño de cada clúster en prueba (verifica ausencia de grupos degenerados):
+----------+------+
|prediction| count|
+----------+------+
|         0|184365|
|         1|147857|
|         2| 12604|
+----------+------+



## 5 Análisis de resultados

A continuación se analiza cada modelo por separado y luego se hace una lectura transversal, identificando fortalezas y áreas de oportunidad.

### 5.1 Modelo supervisado: árbol de decisión

Resultados obtenidos: la búsqueda con validación seleccionó `maxDepth = 10`, `minInstancesPerNode = 50` y `maxBins = 32`, con un F1 ponderado de 0.7264 en entrenamiento y 0.7254 en prueba (brecha de apenas 0.0010).

Fortalezas:

- Tuvo una generalización sobresaliente. La diferencia entre el F1 de entrenamiento y el de prueba es de 0.001, prácticamente nula. Es la evidencia más sólida del trabajo: la estrategia anti-sobreajuste (validación interna con TrainValidationSplit más pre-poda con `minInstancesPerNode`) funcionó. El modelo no memoriza el conjunto de entrenamiento, sino que captura patrones que se sostienen sobre datos no vistos.
- El ajuste de hiper-parámetros se decantó por una complejidad intermedia (profundidad 10 con poda ligera) en lugar del árbol más profundo disponible, lo que confirma que la regularización por número mínimo de instancias por hoja fue determinante para el equilibrio sesgo-varianza.
- El desempeño (F1 ≈ 0.725) es coherente y reproducible sobre 1.37 millones de registros de entrenamiento, y se logró evitando la fuga de información al excluir las variables de las que se derivó la etiqueta (`High`, `Low`, `Volatilidad_Diaria_Pct`). Que el F1 no sea cercano a 1 es, paradójicamente, una señal de salud: descarta una fuga encubierta.

Áreas de oportunidad:

- El F1 de 0.725 implica que aproximadamente uno de cada cuatro registros se clasifica de forma incorrecta. Buena parte de este techo es intrínseco al problema: la etiqueta `Nivel_Volatilidad` es la discretización de una variable continua mediante umbrales (2 % y 5 %), por lo que los registros cercanos a esas fronteras son ambiguos por naturaleza y limitan el F1 alcanzable, independientemente del modelo.
- La evaluación se apoyó únicamente en el F1 ponderado. Para profundizar el diagnóstico convendría incorporar la matriz de confusión y el recall por clase, que revelarían si el error se concentra entre clases adyacentes y si la clase minoritaria (`Alta`) se predice peor por el desbalance de estratos. También la importancia de variables ayudaría a interpretar qué impulsa la predicción.
- Como líneas de mejora del desempeño podrían explorarse modelos de conjunto (Random Forest o Gradient-Boosted Trees), un enfoque ordinal que respete el orden Baja < Media < Alta, o el uso de pesos por clase para compensar el desbalance.

### 5.2 Modelo no supervisado: K-Means

Resultados obtenidos: el barrido de k (con costo y silueta) y la regla de selección automática eligieron `k = 3`, el valor con mayor silueta entre las soluciones no degeneradas. Sobre el conjunto de prueba la silueta fue 0.5886 y los tres clústeres quedaron con 184,365, 147,857 y 12,604 instancias (aproximadamente 53 %, 43 % y 4 %).

Fortalezas:

- El ajuste adicional (transformación `log1p` sobre precios y volumen, más estandarización) resolvió el problema de agrupamientos degenerados del enfoque previo. Antes, la fuerte asimetría de los precios producía un clúster dominante, otro diminuto y hasta clústeres vacíos, acompañados de una silueta engañosamente alta. Ahora los tres grupos tienen tamaños sustanciales y ninguno cae por debajo del umbral de degeneración.
- La regla de selección de k demostró su utilidad: descartó k = 4, 6 y 7, que pese a presentar siluetas competitivas escondían clústeres de un solo punto, y se quedó con k = 3, que combina buena cohesión-separación con grupos balanceados. Esto evita el análogo no supervisado del sobreajuste (un k inflado que aísla outliers).
- La silueta en prueba (0.5886) es del mismo orden que la obtenida durante el ajuste, lo que indica que la estructura de clústeres no es un artefacto del entrenamiento, sino que se reproduce en datos nuevos.

Áreas de oportunidad:

- Una silueta de ~0.52 a 0.59 corresponde a una estructura moderada: los grupos existen pero presentan solapamiento. Es un resultado honesto y útil, lejos del 0.99 espurio anterior, pero todavía hay margen para una separación más nítida.
- Falta perfilar los clústeres. El siguiente paso natural es calcular la media de las variables originales por grupo para asignarles un significado de negocio (por ejemplo, confirmar si el clúster minoritario del 4 % agrupa acciones de precio alto y baja liquidez). Sin ese perfilado, la segmentación es estadísticamente válida pero aún poco accionable.
- Podrían explorarse alternativas que capturen mejor la forma de los datos: un escalador robusto, modelos de mezcla gaussiana (GMM) que admiten clústeres no esféricos, u otras métricas de distancia. El umbral de degeneración del 1 % es además una heurística que conviene documentar como decisión de diseño.

### 5.3 Lectura transversal y manejo de grandes volúmenes

- Calidad de la muestra y de las particiones. La diferencia de pesos entre la población y la muestra M es prácticamente cero en todos los estratos, y la división Train/Test conservó la proporción 80/20 en cada uno de ellos (todos entre 79 % y 81 %), con conjuntos disjuntos y unión igual a M. Esta ausencia de sesgo es la que da confianza en que las métricas reportadas reflejan el comportamiento real sobre la población y no un artefacto del muestreo.
- Contraste entre paradigmas. El árbol de decisión, basado en cortes por umbral, resultó robusto y casi insensible a la escala y a la asimetría de las variables. K-Means, basado en distancias euclidianas, exigió transformaciones previas para no degenerar. El ejercicio confirma, sobre los mismos datos, que los métodos por distancia son intrínsecamente más sensibles a la calidad y a la forma de los datos que los métodos por partición.
- Escalabilidad. Todo el proceso (muestreo, partición, ajuste de hiper-parámetros y evaluación) se ejecutó en PySpark sobre 1.72 millones de registros extraídos de los 34.6 millones originales, empleando únicamente métricas y evaluadores distribuibles, en línea con el criterio de grandes volúmenes definido en la sección 3.

### 5.4 Conclusión

El modelo supervisado es el resultado más sólido, generaliza casi sin pérdida de desempeño y su techo está marcado por la ambigüedad propia de discretizar una variable continua, no por una limitación del entrenamiento. El modelo no supervisado pasó de inservible a útil gracias al pre-procesamiento, y hoy entrega tres segmentos balanceados con una calidad moderada; su principal tarea pendiente es el perfilado interpretativo. En conjunto, el trabajo evidencia que, con grandes volúmenes de datos, la calidad de los resultados depende tanto del algoritmo como del rigor con que se construyen la muestra, las particiones y el pre-procesamiento.